In [108]:
import pandas as pd
df = pd.read_csv("data/processed/processed_df.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [13]:
df.shape

(7032, 20)

In [72]:
# --- Billing / Value features ---
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'].replace(0, 1))
df['charge_gap'] = df['MonthlyCharges'] - df['avg_monthly_spend']
df['is_high_value'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

In [5]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [73]:
# --- Tenure band (categorical) ---
df['tenure_band'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 36, 72],
    labels=['0-12', '12-36', '36+'],
    right=True
)
df['tenure_band'] = df['tenure_band'].astype(str)

In [16]:
df.groupby('tenure_band')['Churn'].mean()

tenure_band
0-12     0.476782
12-36    0.255388
36+      0.119294
Name: Churn, dtype: float64

In [17]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,avg_monthly_spend,charge_gap,is_high_value,tenure_band
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,0,29.850000,0.000000,0,0-12
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,0,55.573529,1.376471,0,12-36
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,1,54.075000,-0.225000,0,0-12
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,0,40.905556,1.394444,0,36+
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,1,75.825000,-5.125000,1,0-12


In [74]:
# --- Service usage counts (numeric) ---
streamning_services = ['StreamingTV','StreamingMovies']
df['streaming_count'] = ((df[streamning_services] == 'Yes') | (df[streamning_services] == 'yes')).sum(axis=1)

security_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df['security_count'] = ((df[security_services] == 'Yes') | (df[security_services] == 'yes')).sum(axis=1)   

In [19]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,avg_monthly_spend,charge_gap,is_high_value,tenure_band,streaming_count,security_count
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,...,Electronic check,29.85,29.85,0,29.850000,0.000000,0,0-12,0,1
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,...,Mailed check,56.95,1889.50,0,55.573529,1.376471,0,12-36,0,2
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,...,Mailed check,53.85,108.15,1,54.075000,-0.225000,0,0-12,0,2
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,...,Bank transfer (automatic),42.30,1840.75,0,40.905556,1.394444,0,36+,0,3
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,...,Electronic check,70.70,151.65,1,75.825000,-5.125000,1,0-12,0,0


In [75]:
df['month_to_month_paperless'] = (
    (df['Contract'] == 'Month-to-month') &
    (df['PaperlessBilling'] == 'Yes')
).astype(int)

df['payment_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

In [21]:
df.groupby('month_to_month_paperless')['Churn'].mean()

month_to_month_paperless
0    0.139451
1    0.482985
Name: Churn, dtype: float64

In [22]:
df.groupby('payment_electronic_check')['Churn'].mean()

payment_electronic_check
0    0.170988
1    0.452854
Name: Churn, dtype: float64

In [23]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TotalCharges,Churn,avg_monthly_spend,charge_gap,is_high_value,tenure_band,streaming_count,security_count,month_to_month_paperless,payment_electronic_check
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,...,29.85,0,29.850000,0.000000,0,0-12,0,1,1,1
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,...,1889.50,0,55.573529,1.376471,0,12-36,0,2,0,0
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,...,108.15,1,54.075000,-0.225000,0,0-12,0,2,1,0
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,...,1840.75,0,40.905556,1.394444,0,36+,0,3,0,0
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,...,151.65,1,75.825000,-5.125000,1,0-12,0,0,1,1


In [76]:
    # --- Vulnerability / persona flags (numeric) ---
df['no_support_services'] = ((df['TechSupport'] == 'No') &(df['OnlineSecurity'] == 'No')).astype(int)
    
df['is_isolated'] = ((df['Partner'] == 'No') &(df['Dependents'] == 'No')).astype(int)
    
df['fiber_no_security'] = ((df['InternetService'] == 'Fiber optic') &(df['OnlineSecurity'] == 'No')).astype(int)
    
df['no_internet_services'] = (df['InternetService'] == 'No').astype(int)

In [25]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,is_high_value,tenure_band,streaming_count,security_count,month_to_month_paperless,payment_electronic_check,no_support_services,is_isolated,fiber_no_security,no_internet_services
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,...,0,0-12,0,1,1,1,1,0,0,0
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,...,0,12-36,0,2,0,0,0,1,0,0
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,...,0,0-12,0,2,1,0,0,1,0,0
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,...,0,36+,0,3,0,0,0,1,0,0
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,...,1,0-12,0,0,1,1,1,1,1,0


In [78]:
df.columns,len(df.columns)

(Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges', 'Churn', 'avg_monthly_spend',
        'charge_gap', 'is_high_value', 'tenure_band', 'streaming_count',
        'security_count', 'month_to_month_paperless',
        'payment_electronic_check', 'no_support_services', 'is_isolated',
        'fiber_no_security', 'no_internet_services'],
       dtype='str'),
 32)

saving base + engineered features

In [ ]:
# import os

# # create directory if it doesn't exist
# os.makedirs("data/processed", exist_ok=True)
# # save dataframe
# df.to_csv("data/processed/base_plus_engineered.csv", index=False)

choosing segmentation features

In [79]:
# Choose segmentation features (mixed)
segmentation_features = [
    # Demographics (categorical)
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    # Lifecycle (numeric + categorical)
    'tenure', 'tenure_band',
    # Value (numeric)
    'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'charge_gap', 'is_high_value',
    # Services (mixed)
    'PhoneService', 'MultipleLines', 'InternetService',
    'streaming_count', 'security_count',
    # Contract / payment (mixed)
    'Contract', 'PaperlessBilling', 'PaymentMethod',
    'payment_electronic_check',
    'month_to_month_paperless',
    # Risk flags (numeric)
    'no_support_services', 'is_isolated', 'fiber_no_security',
    'no_internet_services'
]

data_seg = df[segmentation_features].copy()

In [80]:
data_seg.shape

(7032, 25)

saving engineered feature for segementation

In [ ]:
# import os

# # create directory if it doesn't exist
# os.makedirs("data/processed", exist_ok=True)

# # save dataframe
# data_seg.to_csv("data/processed/segment_modelling_features.csv", index=False)

segmentation modelling

In [81]:
pd.set_option('display.max_columns', None)

In [82]:
data_seg = pd.read_csv("data/processed/segment_modelling_features.csv")
data_seg.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,tenure_band,MonthlyCharges,TotalCharges,avg_monthly_spend,charge_gap,is_high_value,PhoneService,MultipleLines,InternetService,streaming_count,security_count,Contract,PaperlessBilling,PaymentMethod,payment_electronic_check,month_to_month_paperless,no_support_services,is_isolated,fiber_no_security,no_internet_services
0,Female,no,Yes,No,1,0-12,29.85,29.85,29.850000,0.000000,0,No,No phone service,DSL,0,1,Month-to-month,Yes,Electronic check,1,1,1,0,0,0
1,Male,no,No,No,34,12-36,56.95,1889.50,55.573529,1.376471,0,Yes,No,DSL,0,2,One year,No,Mailed check,0,0,0,1,0,0
2,Male,no,No,No,2,0-12,53.85,108.15,54.075000,-0.225000,0,Yes,No,DSL,0,2,Month-to-month,Yes,Mailed check,0,1,0,1,0,0
3,Male,no,No,No,45,36+,42.30,1840.75,40.905556,1.394444,0,No,No phone service,DSL,0,3,One year,No,Bank transfer (automatic),0,0,0,1,0,0
4,Female,no,No,No,2,0-12,70.70,151.65,75.825000,-5.125000,1,Yes,No,Fiber optic,0,0,Month-to-month,Yes,Electronic check,1,1,1,1,1,0


In [83]:
data_seg.dtypes

gender                          str
SeniorCitizen                   str
Partner                         str
Dependents                      str
tenure                        int64
tenure_band                     str
MonthlyCharges              float64
TotalCharges                float64
avg_monthly_spend           float64
charge_gap                  float64
is_high_value                 int64
PhoneService                    str
MultipleLines                   str
InternetService                 str
streaming_count               int64
security_count                int64
Contract                        str
PaperlessBilling                str
PaymentMethod                   str
payment_electronic_check      int64
month_to_month_paperless      int64
no_support_services           int64
is_isolated                   int64
fiber_no_security             int64
no_internet_services          int64
dtype: object

In [84]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges","avg_monthly_spend","charge_gap","streaming_count","security_count"]
cat_cols = [c for c in data_seg.columns if c not in num_cols]

num_cols, cat_cols

(['tenure',
  'MonthlyCharges',
  'TotalCharges',
  'avg_monthly_spend',
  'charge_gap',
  'streaming_count',
  'security_count'],
 ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure_band',
  'is_high_value',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'payment_electronic_check',
  'month_to_month_paperless',
  'no_support_services',
  'is_isolated',
  'fiber_no_security',
  'no_internet_services'])

In [85]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_seg[num_cols] = scaler.fit_transform(data_seg[num_cols])

In [86]:
data_seg.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,tenure_band,MonthlyCharges,TotalCharges,avg_monthly_spend,charge_gap,is_high_value,PhoneService,MultipleLines,InternetService,streaming_count,security_count,Contract,PaperlessBilling,PaymentMethod,payment_electronic_check,month_to_month_paperless,no_support_services,is_isolated,fiber_no_security,no_internet_services
0,Female,no,Yes,No,-1.280248,0-12,-1.161694,-0.994194,-1.157889,0.000465,0,No,No phone service,DSL,-0.906251,-0.206314,Month-to-month,Yes,Electronic check,1,1,1,0,0,0
1,Male,no,No,No,0.064303,12-36,-0.260878,-0.173740,-0.305658,0.526643,0,Yes,No,DSL,-0.906251,0.571179,One year,No,Mailed check,0,0,0,1,0,0
2,Male,no,No,No,-1.239504,0-12,-0.363923,-0.959649,-0.355305,-0.085545,0,Yes,No,DSL,-0.906251,0.571179,Month-to-month,Yes,Mailed check,0,1,0,1,0,0
3,Male,no,No,No,0.512486,36+,-0.747850,-0.195248,-0.791614,0.533513,0,No,No phone service,DSL,-0.906251,1.348672,One year,No,Bank transfer (automatic),0,0,0,1,0,0
4,Female,no,No,No,-1.239504,0-12,0.196178,-0.940457,0.365282,-1.958649,1,Yes,No,Fiber optic,-0.906251,-0.983807,Month-to-month,Yes,Electronic check,1,1,1,1,1,0


In [87]:
for c in cat_cols:
    data_seg[c] = data_seg[c].astype(str)

cat_idx = [data_seg.columns.get_loc(c) for c in cat_cols]
cat_idx

[0, 1, 2, 3, 5, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 24]

In [88]:
# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE VALIDATION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def validate_feature_consistency(df, expected_num_cols, expected_cat_cols, phase=""):
    """
    Validate that a dataframe has expected features with correct properties.
    
    Args:
        df: DataFrame to validate
        expected_num_cols: list of expected numeric feature names
        expected_cat_cols: list of expected categorical feature names
        phase: str, phase name for error messages (e.g., 'training', 'prediction')
    
    Returns:
        dict with validation results
    
    Raises:
        ValueError: if features don't match expectations
    """
    results = {
        'phase': phase,
        'all_pass': True,
        'checks': {}
    }
    
    # Check 1: All expected numeric columns present
    missing_num = [c for c in expected_num_cols if c not in df.columns]
    if missing_num:
        results['all_pass'] = False
        results['checks']['missing_numeric'] = missing_num
        raise ValueError(f"[{phase}] Missing numeric columns: {missing_num}")
    results['checks']['numeric_present'] = '✓'
    
    # Check 2: All expected categorical columns present
    missing_cat = [c for c in expected_cat_cols if c not in df.columns]
    if missing_cat:
        results['all_pass'] = False
        results['checks']['missing_categorical'] = missing_cat
        raise ValueError(f"[{phase}] Missing categorical columns: {missing_cat}")
    results['checks']['categorical_present'] = '✓'
    
    # Check 3: No extra columns (only expected features)
    expected_cols = set(expected_num_cols + expected_cat_cols)
    actual_cols = set(df.columns) & expected_cols  # Only check expected columns
    extra_cols = [c for c in df.columns if c not in expected_cols]
    if extra_cols:
        results['checks']['extra_columns_warning'] = f"⚠ Extra columns in data: {extra_cols[:3]}..."
    
    # Check 4: Verify numeric columns are numeric
    for col in expected_num_cols:
        if col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                results['all_pass'] = False
                raise ValueError(f"[{phase}] Numeric column '{col}' has dtype {df[col].dtype}, expected numeric")
    results['checks']['numeric_dtypes'] = '✓'
    
    # Check 5: Verify categorical columns are string/object type
    non_string_cats = []
    for col in expected_cat_cols:
        if col in df.columns:
            # Allow 'object' and string dtypes, but warn if 'int' or 'float'
            if df[col].dtype not in ['object', 'string']:
                non_string_cats.append(f"{col} ({df[col].dtype})")
    
    if non_string_cats:
        results['checks']['categorical_dtype_warning'] = f"⚠ Categorical columns not string dtype: {non_string_cats}"
    else:
        results['checks']['categorical_dtypes'] = '✓'
    
    # Check 6: Feature count consistency
    results['checks']['numeric_count'] = len(expected_num_cols)
    results['checks']['categorical_count'] = len(expected_cat_cols)
    
    return results


def print_feature_validation_report(results_training, results_prediction=None):
    """
    Print a formatted validation report comparing training and prediction features.
    """
    print("\n" + "="*80)
    print("FEATURE CONSISTENCY VALIDATION REPORT")
    print("="*80)
    
    print(f"\n[TRAINING PHASE]")
    print(f"  Numeric features: {results_training['checks']['numeric_count']}")
    print(f"  Categorical features: {results_training['checks']['categorical_count']}")
    print(f"  Status: {results_training['checks']}")
    
    if results_prediction:
        print(f"\n[PREDICTION PHASE]")
        print(f"  Numeric features: {results_prediction['checks']['numeric_count']}")
        print(f"  Categorical features: {results_prediction['checks']['categorical_count']}")
        print(f"  Status: {results_prediction['checks']}")
        
        # Compare
        num_match = results_training['checks']['numeric_count'] == results_prediction['checks']['numeric_count']
        cat_match = results_training['checks']['categorical_count'] == results_prediction['checks']['categorical_count']
        
        print(f"\n[COMPARISON]")
        print(f"  Numeric count match: {'✓ YES' if num_match else '✗ NO'}")
        print(f"  Categorical count match: {'✓ YES' if cat_match else '✗ NO'}")
        print(f"  Overall consistency: {'✓ PASS' if (num_match and cat_match and results_prediction['all_pass']) else '✗ FAIL'}")
    
    print("="*80 + "\n")

In [89]:
# ── Validate training features ─────────────────────────────────────────────────
print("Validating training phase features...")
training_validation = validate_feature_consistency(data_seg, num_cols, cat_cols, phase="TRAINING")
print(f"✓ Training features validated: {num_cols + cat_cols}")
print(f"  - Numeric: {len(num_cols)} features")
print(f"  - Categorical: {len(cat_cols)} features")


Validating training phase features...
✓ Training features validated: ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'charge_gap', 'streaming_count', 'security_count', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure_band', 'is_high_value', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'payment_electronic_check', 'month_to_month_paperless', 'no_support_services', 'is_isolated', 'fiber_no_security', 'no_internet_services']
  - Numeric: 7 features
  - Categorical: 18 features


In [90]:
from kmodes.kprototypes import KPrototypes
import matplotlib.pyplot as plt

In [91]:
# ── Convert to numpy array for K-Prototypes ───────────────────────────────────
X = data_seg.to_numpy()
print(f"Training data shape: {X.shape} (rows × columns)")
print(f"Categorical indices: {cat_idx}")


Training data shape: (7032, 25) (rows × columns)
Categorical indices: [0, 1, 2, 3, 5, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [39]:
# X = data_seg.to_numpy()
# costs = []
# ks = range(2, 11)

# for k in ks:
#     kproto = KPrototypes(n_clusters=k, init="Cao", n_init=5, verbose=0, random_state=42)
#     labels = kproto.fit_predict(X, categorical=cat_idx)
#     costs.append(kproto.cost_)

# plt.figure(figsize=(8, 4))
# plt.plot(list(ks), costs, marker="o")
# plt.xlabel("Number of clusters (k)")
# plt.ylabel("Cost")
# plt.title("K-Prototypes Elbow Plot")
# plt.grid(True, alpha=0.3)
# plt.show()

KeyboardInterrupt: 

In [92]:
final_k = 4

kproto = KPrototypes(
    n_clusters=final_k,
    init="Cao",
    n_init=10,
    verbose=0,
    random_state=42
)

cluster_labels = kproto.fit_predict(X, categorical=cat_idx)
data_seg["segment"] = cluster_labels


In [93]:
data_seg['segment'].value_counts()

segment
1    2099
3    1951
0    1697
2    1285
Name: count, dtype: int64

In [94]:
data_seg.shape

(7032, 26)

adding labels to the actual processed data

In [95]:
import pandas as pd
df = pd.read_csv("data/processed/processed_df.csv")
df.shape

(7032, 20)

In [96]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [97]:
df["segment"] = cluster_labels

In [99]:
df.shape

(7032, 21)

In [98]:
segment_summary = df.groupby("segment").agg(
    customers=("segment", "size"),
    churn_rate=("Churn", lambda x: (x == 1).mean()),
    tenure_median=("tenure", "median"),
    monthly_median=("MonthlyCharges", "median"),
    total_median=("TotalCharges", "median")
).sort_values("customers", ascending=False)

segment_summary

,customers,churn_rate,tenure_median,monthly_median,total_median
segment,,,,,
1,2099,0.162935,16.0,20.60,402.60
3,1951,0.562276,10.0,81.10,834.10
0,1697,0.156158,63.0,98.70,5676.65
2,1285,0.128405,38.0,59.75,2203.65


In [100]:
segment_labels = {
    3: "At risk High-value 🚨",
    0: "Loyal High-Value 💎",
    2: "Stable Mid-Value 👍",
    1: "Low Engagement ⚠️"
}

df['segment_label'] = df['segment'].map(segment_labels)

In [101]:
df.shape

(7032, 22)

In [103]:
df['segment_label'].value_counts()

segment_label
Low Engagement ⚠️       2099
At risk High-value 🚨    1951
Loyal High-Value 💎      1697
Stable Mid-Value 👍      1285
Name: count, dtype: int64

In [104]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,segment,segment_label
0,Female,no,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,1,Low Engagement ⚠️
1,Male,no,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,2,Stable Mid-Value 👍
2,Male,no,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,2,Stable Mid-Value 👍
3,Male,no,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,2,Stable Mid-Value 👍
4,Female,no,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,3,At risk High-value 🚨


saving processed data with segment labels

In [ ]:
# import os

# os.makedirs("data/processed", exist_ok=True)
# df.to_csv("data/processed/df_with_segment_labels.csv", index=False)

saving model artifact

In [105]:
import joblib
import os

# Single consistent directory for all model artifacts
os.makedirs("../models/segmentation", exist_ok=True)

# Save the K-Prototypes model
joblib.dump(kproto, "../models/segmentation/kproto_segmentation_model.pkl")

# Save the scaler
joblib.dump(scaler, "../models/segmentation/scaler.pkl")

print("Model and scaler saved successfully!")

Model and scaler saved successfully!


verify prediction

In [106]:
import joblib

# ── Load model artifacts ──────────────────────────────────────────
kproto = joblib.load("../models/segmentation/kproto_segmentation_model.pkl")
scaler  = joblib.load("../models/segmentation/scaler.pkl")

print("Model and scaler loaded successfully!")

Model and scaler loaded successfully!


(['tenure',
  'MonthlyCharges',
  'TotalCharges',
  'avg_monthly_spend',
  'charge_gap',
  'streaming_count',
  'security_count'],
 ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure_band',
  'is_high_value',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'payment_electronic_check',
  'month_to_month_paperless',
  'no_support_services',
  'is_isolated',
  'fiber_no_security',
  'no_internet_services'])

In [107]:
# ── Define columns (must match training) ─────────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
            'avg_monthly_spend', 'charge_gap', 'streaming_count', 'security_count']

cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure_band',
            'is_high_value', 'PhoneService', 'MultipleLines', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod',
            'payment_electronic_check', 'month_to_month_paperless',
            'no_support_services', 'is_isolated', 'fiber_no_security', 'no_internet_services']

# ── Load processed data for testing ──────────────────────────────
# notebooks\data\processed\segment_modelling_features.csv
dataseg = pd.read_csv("data/processed/segment_modelling_features.csv")

# ── VALIDATION: Check feature consistency ────────────────────────
print("\n" + "─"*80)
print("VALIDATING TEST DATA FEATURES...")
print("─"*80)
try:
    # Validate all features exist and have correct properties
    test_validation = validate_feature_consistency(dataseg, num_cols, cat_cols, phase="PREDICTION")
    print(f"✓ Test data feature check PASSED")
    print(f"  - Numeric columns: {len(num_cols)} ✓")
    print(f"  - Categorical columns: {len(cat_cols)} ✓")
    
    # Ensure categorical columns are strings (required for K-Prototypes prediction)
    print("\nConverting categorical columns to string type...")
    for c in cat_cols:
        if c in dataseg.columns:
            dataseg[c] = dataseg[c].astype(str)
    print(f"✓ Categorical columns converted to string type")
    
except ValueError as e:
    print(f"✗ VALIDATION FAILED: {e}")
    print("  Cannot proceed with prediction. Check features in test data.")
    raise

# ── Preprocess: scale numeric cols only ──────────────────────────
dataseg[num_cols] = scaler.transform(dataseg[num_cols])

# ── Get categorical column indices ───────────────────────────────
cat_idx = [dataseg.columns.get_loc(c) for c in cat_cols]

# ── Predict ───────────────────────────────────────────────────────
X = dataseg.to_numpy()
print(f"\nTest data shape: {X.shape} (rows × columns)")
print(f"Categorical indices: {cat_idx}")

labels = kproto.predict(X, categorical=cat_idx)

# ── Map to readable segment labels ────────────────────────────────
segment_labels = {
    3: 'At risk High-value',
    0: 'Loyal High-Value',
    2: 'Stable Mid-Value',
    1: 'Low Engagement'
}

dataseg['segment']       = labels
dataseg['segment_label'] = dataseg['segment'].map(segment_labels)

print("\n" + "─"*80)
print("SEGMENT LABEL DISTRIBUTION (Feature validation ✓ passed)")
print("─"*80)
print(dataseg['segment_label'].value_counts())
print("─"*80 + "\n")

dataseg.head()


────────────────────────────────────────────────────────────────────────────────
VALIDATING TEST DATA FEATURES...
────────────────────────────────────────────────────────────────────────────────
✓ Test data feature check PASSED
  - Numeric columns: 7 ✓
  - Categorical columns: 18 ✓

Converting categorical columns to string type...
✓ Categorical columns converted to string type

Test data shape: (7032, 25) (rows × columns)
Categorical indices: [0, 1, 2, 3, 5, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 24]

────────────────────────────────────────────────────────────────────────────────
SEGMENT LABEL DISTRIBUTION (Feature validation ✓ passed)
────────────────────────────────────────────────────────────────────────────────
segment_label
Low Engagement        2099
At risk High-value    1951
Loyal High-Value      1697
Stable Mid-Value      1285
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────



,gender,SeniorCitizen,Partner,Dependents,tenure,tenure_band,MonthlyCharges,TotalCharges,avg_monthly_spend,charge_gap,is_high_value,PhoneService,MultipleLines,InternetService,streaming_count,security_count,Contract,PaperlessBilling,PaymentMethod,payment_electronic_check,month_to_month_paperless,no_support_services,is_isolated,fiber_no_security,no_internet_services,segment,segment_label
0,Female,no,Yes,No,-1.280248,0-12,-1.161694,-0.994194,-1.157889,0.000465,0,No,No phone service,DSL,-0.906251,-0.206314,Month-to-month,Yes,Electronic check,1,1,1,0,0,0,1,Low Engagement
1,Male,no,No,No,0.064303,12-36,-0.260878,-0.173740,-0.305658,0.526643,0,Yes,No,DSL,-0.906251,0.571179,One year,No,Mailed check,0,0,0,1,0,0,2,Stable Mid-Value
2,Male,no,No,No,-1.239504,0-12,-0.363923,-0.959649,-0.355305,-0.085545,0,Yes,No,DSL,-0.906251,0.571179,Month-to-month,Yes,Mailed check,0,1,0,1,0,0,2,Stable Mid-Value
3,Male,no,No,No,0.512486,36+,-0.747850,-0.195248,-0.791614,0.533513,0,No,No phone service,DSL,-0.906251,1.348672,One year,No,Bank transfer (automatic),0,0,0,1,0,0,2,Stable Mid-Value
4,Female,no,No,No,-1.239504,0-12,0.196178,-0.940457,0.365282,-1.958649,1,Yes,No,Fiber optic,-0.906251,-0.983807,Month-to-month,Yes,Electronic check,1,1,1,1,1,0,3,At risk High-value
